In [ ]:
!pip install ultralytics roboflow

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="my_api_key")
project = rf.workspace("zhiyaoians-workspace").project("flappy-vhl19")
dataset = project.version(1).download("yolov11")
print(f"Downloaded to: {dataset.location}")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Flappy-1 in yolov11:: 100%|██████████| 73/73 [00:00<00:00, 9038.65it/s]

Downloaded to: /content/Flappy-1


In [ ]:
import os, random, shutil
from pathlib import Path
from ultralytics import YOLO

DATASET_DIR = dataset.location
VAL_RATIO   = 0.2
EPOCHS      = 150
BATCH       = 8
PATIENCE    = 30

root = Path(DATASET_DIR)
train_imgs = root / "train" / "images"
train_lbls = root / "train" / "labels"
val_imgs   = root / "valid" / "images"
val_lbls   = root / "valid" / "labels"

if not val_imgs.exists() or len(list(val_imgs.glob("*.*"))) == 0:
    all_imgs = sorted([
        f for f in train_imgs.glob("*.*")
        if f.suffix.lower() in (".jpg", ".jpeg", ".png")
        and (train_lbls / f"{f.stem}.txt").exists()
    ])
    random.seed(42)
    random.shuffle(all_imgs)
    n_val = max(1, int(len(all_imgs) * VAL_RATIO))

    val_imgs.mkdir(parents=True, exist_ok=True)
    val_lbls.mkdir(parents=True, exist_ok=True)

    for img in all_imgs[:n_val]:
        lbl = train_lbls / f"{img.stem}.txt"
        shutil.move(str(img), str(val_imgs / img.name))
        shutil.move(str(lbl), str(val_lbls / lbl.name))

    print(f"Split: {len(all_imgs) - n_val} train, {n_val} val")
else:
    print("Val split already exists")

yaml_path = root / "data.yaml"
yaml_path.write_text(f"""path: {root.resolve()}
train: train/images
val: valid/images

nc: 2
names: ['bird', 'pipe']
""")

model = YOLO("yolo11n.pt")

model.train(
    data=str(yaml_path),
    epochs=EPOCHS,
    batch=BATCH,
    imgsz=640,
    patience=PATIENCE,
    augment=True,
    hsv_h=0.02, hsv_s=0.5, hsv_v=0.3,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    scale=0.3,
    translate=0.2,
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.001,
    project="runs/flappy",
    name="yolo11n",
    exist_ok=True,
)

metrics = model.val(data=str(yaml_path))
print(f"\n📊 mAP50: {metrics.box.map50:.3f} | mAP50-95: {metrics.box.map:.3f}")

# ── Export ONNX ────────────────────────
model.export(format="onnx", imgsz=640)


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ Val split already exists
✅ data.yaml updated
Ultralytics 8.4.33 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Flappy-1/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import files
files.download("/content/runs/detect/runs/flappy/yolo11n/weights")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>